In [ ]:
import subprocess, sys, os, importlib

# -- Setup path and install dependencies ---------------------------
sys.path.insert(0, os.path.join(os.getcwd(), 'src'))

# -- Force reload all project modules ------------------------------
for mod_name in ['swing_equation', 'data_generator', 'model', 'loss', 'trainer', 'stability_analysis']:
    if mod_name in sys.modules:
        importlib.reload(sys.modules[mod_name])

print('[OK] Modules loaded successfully')

In [ ]:
import torch
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap

from swing_equation import SMIBParameters, SwingEquationSolver
from data_generator import PINNDataGenerator
from model import PINN, count_parameters
from stability_analysis import StabilityAnalyzer

# Setup device
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')

In [ ]:
# -- Load trained PINN model --------------------------------------
MODEL_PATH = 'models/pinn_trained-1.pt'

if os.path.exists(MODEL_PATH):
    checkpoint = torch.load(MODEL_PATH, map_location=device)
    model = PINN(n_hidden_layers=4, n_neurons=64)
    model.load_state_dict(checkpoint['model_state_dict'])
    model.to(device)
    model.eval()
    print(f'[OK] Loaded trained model from {MODEL_PATH}')
    print(f'     Trainable parameters: {count_parameters(model):,}')
else:
    print(f'[ERROR] Model file not found: {MODEL_PATH}')
    print('Please train the model first using the training notebook.')
    raise FileNotFoundError(f'Model not found: {MODEL_PATH}')

In [ ]:
# -- Setup physics parameters (should match training) ------------
params = SMIBParameters(H=5.0, D=0.05, Pm=0.8, Pmax=2.1)
print(f'[OK] Physics parameters configured')
print(f'     Equilibrium angle: {np.degrees(params.delta_eq):.2f} degrees')
print(f'     Synchronous speed: {params.omega0:.2f} rad/s')

In [ ]:
# -- Initialize data generator and solver ------------------------
gen = PINNDataGenerator(params, seed=42)
solver = SwingEquationSolver(params)
analyzer = StabilityAnalyzer(model, params, device)
print('[OK] Data generator, solver, and analyzer initialized')

In [ ]:
# -- Generate RK4 reference trajectory ---------------------------
FAULT_START = 0.1
FAULT_END = 0.2
T_TOTAL = 2.0

traj_rk4 = gen.generate_reference_trajectory(
    fault_start=FAULT_START,
    fault_end=FAULT_END,
    t_total=T_TOTAL
)

print(f'[OK] RK4 reference trajectory generated')
print(f'     Time span: 0 to {T_TOTAL}s')
print(f'     Fault window: {FAULT_START}s to {FAULT_END}s')
print(f'     Number of points: {len(traj_rk4["t"])}')

In [ ]:
# -- Get PINN predictions ----------------------------------------
# Use the same time points as RK4 for fair comparison
t_eval = traj_rk4['t']

# Get PINN predictions (returns delta and omega - both absolute)
delta_pinn, omega_pinn = analyzer.predict(t_eval)

# Extract RK4 data
delta_rk4 = traj_rk4['delta']
omega_rk4 = traj_rk4['omega']

print(f'[OK] PINN predictions computed')
print(f'     Time points: {len(t_eval)}')
print(f'     Delta range (RK4): {np.degrees(delta_rk4.min()):.2f}° to {np.degrees(delta_rk4.max()):.2f}°')
print(f'     Delta range (PINN): {np.degrees(delta_pinn.min()):.2f}° to {np.degrees(delta_pinn.max()):.2f}°')

In [ ]:
# -- Compute accuracy metrics ------------------------------------
mae_delta = np.mean(np.abs(delta_pinn - delta_rk4))
mae_omega = np.mean(np.abs(omega_pinn - omega_rk4))
rmse_delta = np.sqrt(np.mean((delta_pinn - delta_rk4)**2))
rmse_omega = np.sqrt(np.mean((omega_pinn - omega_rk4)**2))

print('\n=== Accuracy Metrics ===')
print(f'MAE delta: {np.degrees(mae_delta):.4f} degrees')
print(f'MAE omega: {mae_omega:.4f} rad/s')
print(f'RMSE delta: {np.degrees(rmse_delta):.4f} degrees')
print(f'RMSE omega: {rmse_omega:.4f} rad/s')
print('========================\n')

## Phase Portrait: Delta vs Omega Comparison

This plot shows the phase portrait (delta vs omega) comparing the RK4 solver (reference) and PINN predictions.

In [ ]:
# -- Create phase portrait comparison plot ----------------------
plt.rcParams['font.family'] = 'sans-serif'
plt.rcParams['text.color'] = '#333333'

fig, ax = plt.subplots(figsize=(12, 10), facecolor='#FDFDFD')
ax.set_facecolor('#FDFDFD')

# Plot RK4 trajectory (reference)
ax.plot(np.degrees(delta_rk4), omega_rk4,
        'b-', linewidth=3, label='RK4 Solver (Reference)', 
        alpha=0.8, zorder=3)

# Plot PINN trajectory
ax.plot(np.degrees(delta_pinn), omega_pinn,
        'r--', linewidth=2.5, label='PINN Prediction',
        alpha=0.8, zorder=2)

# Mark start and end points
ax.plot(np.degrees(delta_rk4[0]), omega_rk4[0],
        'go', markersize=10, label='Start Point', zorder=5, markeredgecolor='darkgreen')
ax.plot(np.degrees(delta_rk4[-1]), omega_rk4[-1],
        'mo', markersize=10, label='End Point', zorder=5, markeredgecolor='darkmagenta')

# Mark equilibrium point
eq_delta = np.degrees(params.delta_eq)
ax.plot(eq_delta, params.omega0, 'k+', markersize=15, markeredgewidth=2, 
        label='Equilibrium', zorder=4)

# Add equilibrium lines
ax.axhline(params.omega0, color='gray', linewidth=1, linestyle=':', alpha=0.5)
ax.axvline(eq_delta, color='gray', linewidth=1, linestyle=':', alpha=0.5)

# Highlight fault region on the trajectory
fault_idx_start = np.argmin(np.abs(t_eval - FAULT_START))
fault_idx_end = np.argmin(np.abs(t_eval - FAULT_END))
ax.plot(np.degrees(delta_rk4[fault_idx_start:fault_idx_end]), 
        omega_rk4[fault_idx_start:fault_idx_end],
        'orange', linewidth=4, alpha=0.3, label='Fault Region', zorder=1)

# Professional axis labels
ax.set_xlabel(r'Rotor Angle $\delta$ (degrees)', fontsize=14, fontweight='bold', 
                labelpad=10, color='#222222')
ax.set_ylabel(r'Angular Velocity $\\omega$ (rad/s)', fontsize=14, fontweight='bold',
                labelpad=10, color='#222222')
ax.set_title('Phase Portrait: RK4 vs PINN Comparison', fontsize=16, fontweight='bold', 
              pad=20, color='#111111')

# Add accuracy metrics as text
textstr = f'RMSE $\delta$: {np.degrees(rmse_delta):.3f}°\nRMSE $\omega$: {rmse_omega:.3f} rad/s'
props = dict(boxstyle='round', facecolor='wheat', alpha=0.8)
ax.text(0.05, 0.95, textstr, transform=ax.transAxes, fontsize=11,
        verticalalignment='top', bbox=props, fontweight='bold')

# Modern styling
ax.grid(True, linestyle='--', linewidth=0.5, color='#E0E0E0', alpha=0.8)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
ax.spines['left'].set_color('#CCCCCC')
ax.spines['bottom'].set_color('#CCCCCC')
ax.tick_params(colors='#555555', labelsize=12)

# Clean legend
ax.legend(loc='best', frameon=True, facecolor='#FFFFFF', 
          edgecolor='#E5E5E5', framealpha=0.95, fontsize=11)

# Save plot
os.makedirs('results/plots', exist_ok=True)
plt.savefig('results/plots/phase_portrait_comparison.png', dpi=300, 
            bbox_inches='tight', facecolor=fig.get_facecolor())

plt.tight_layout()
plt.show()

## Time Series Comparison

Individual time series plots for delta and omega to see the detailed comparison over time.

In [ ]:
# -- Time series comparison plots --------------------------------
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(14, 10), sharex=True)
fig.patch.set_facecolor('#FDFDFD')

# Delta comparison
ax1.plot(t_eval, np.degrees(delta_rk4), 'b-', linewidth=2.5, 
         label='RK4 Solver', alpha=0.8)
ax1.plot(t_eval, np.degrees(delta_pinn), 'r--', linewidth=2, 
         label='PINN Prediction', alpha=0.8)
ax1.fill_between(t_eval, np.degrees(delta_rk4), np.degrees(delta_pinn),
                  alpha=0.15, color='red', label='Error region')
ax1.axvspan(FAULT_START, FAULT_END, alpha=0.2, color='orange', 
           label='Fault Region')
ax1.set_ylabel(r'Rotor Angle $\delta$ (degrees)', fontsize=13, fontweight='bold')
ax1.set_title(f'Delta vs Time - RMSE: {np.degrees(rmse_delta):.4f}°', 
              fontsize=14, fontweight='bold')
ax1.legend(fontsize=11, loc='best')
ax1.grid(True, alpha=0.3)

# Omega comparison
ax2.plot(t_eval, omega_rk4, 'b-', linewidth=2.5,
         label='RK4 Solver', alpha=0.8)
ax2.plot(t_eval, omega_pinn, 'r--', linewidth=2,
         label='PINN Prediction', alpha=0.8)
ax2.fill_between(t_eval, omega_rk4, omega_pinn,
                  alpha=0.15, color='red')
ax2.axvspan(FAULT_START, FAULT_END, alpha=0.2, color='orange')
ax2.set_ylabel(r'Angular Velocity $\\omega$ (rad/s)', fontsize=13, fontweight='bold')
ax2.set_xlabel('Time (s)', fontsize=13, fontweight='bold')
ax2.set_title(f'Omega vs Time - RMSE: {rmse_omega:.4f} rad/s', 
              fontsize=14, fontweight='bold')
ax2.legend(fontsize=11, loc='best')
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('results/plots/time_series_comparison.png', dpi=300, bbox_inches='tight')
plt.show()

## Multiple Fault Scenarios Comparison

Compare phase portraits for different fault durations to see how well the PINN generalizes.

In [ ]:
# -- Generate multiple fault scenarios ---------------------------
fault_durations = [0.05, 0.1, 0.15, 0.2, 0.25]  # Different fault durations
colors = ['#1A85FF', '#40B0A6', '#E1BE6A', '#D41159', '#4B0082']

fig, ax = plt.subplots(figsize=(14, 10), facecolor='#FDFDFD')
ax.set_facecolor('#FDFDFD')

for i, fault_dur in enumerate(fault_durations):
    # Generate RK4 trajectory for this fault duration
    traj = gen.generate_reference_trajectory(
        fault_start=0.1,
        fault_end=0.1 + fault_dur,
        t_total=2.0
    )
    
    # Get PINN predictions
    delta_pinn_i, omega_pinn_i = analyzer.predict(traj['t'])
    
    
    # Plot both trajectories
    label = f'Fault {fault_dur*1000:.0f}ms'
    
    # RK4 (solid lines)
    ax.plot(np.degrees(traj['delta']), traj['omega'],
            color=colors[i], linewidth=2.5, alpha=0.7, 
            linestyle='-', label=f'{label} (RK4)')
    
    # PINN (dashed lines)
    ax.plot(np.degrees(delta_pinn_i), omega_pinn_i,
            color=colors[i], linewidth=2, alpha=0.9,
            linestyle='--', label=f'{label} (PINN)')

# Mark equilibrium
ax.plot(eq_delta, params.omega0, 'k+', markersize=15, markeredgewidth=2, zorder=10)
ax.axhline(params.omega0, color='gray', linewidth=1, linestyle=':', alpha=0.5)
ax.axvline(eq_delta, color='gray', linewidth=1, linestyle=':', alpha=0.5)

ax.set_xlabel(r'Rotor Angle $\delta$ (degrees)', fontsize=14, fontweight='bold')
ax.set_ylabel(r'Angular Velocity $\\omega$ (rad/s)', fontsize=14, fontweight='bold')
ax.set_title('Phase Portraits: Multiple Fault Scenarios (RK4 vs PINN)', 
              fontsize=16, fontweight='bold')

ax.grid(True, linestyle='--', linewidth=0.5, color='#E0E0E0', alpha=0.8)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
ax.tick_params(colors='#555555', labelsize=12)

# Custom legend to distinguish RK4 vs PINN
from matplotlib.lines import Line2D
custom_lines = [Line2D([0], [0], color='black', linewidth=2.5, linestyle='-'),
                Line2D([0], [0], color='black', linewidth=2, linestyle='--')]
ax.legend(custom_lines + [plt.Line2D([0], [0], marker='+', color='black', 
                                     markersize=10, linestyle='None')],
          ['RK4 Solver', 'PINN Prediction', 'Equilibrium'],
          loc='best', fontsize=11, frameon=True, 
          facecolor='#FFFFFF', edgecolor='#E5E5E5')

plt.tight_layout()
plt.savefig('results/plots/multiple_faults_comparison.png', dpi=300, bbox_inches='tight')
plt.show()

print('[OK] Multiple fault scenario comparison completed')

## Summary

This notebook provides comprehensive comparison between the PINN neural network predictions and the RK4 solver results:

1. **Phase Portrait**: Shows the delta vs omega trajectory in phase space, which is crucial for stability analysis
2. **Time Series**: Individual plots for delta and omega over time to visualize temporal accuracy
3. **Multiple Scenarios**: Tests generalization across different fault durations

The accuracy metrics (MAE and RMSE) quantify the agreement between the PINN and the reference RK4 solution.